<h5> Checks whether the time series data is stationary after first differencing, using the ADFuller test (Shifts by 7 days (a week))

In [ ]:
# Imports all necessary packages
import pandas as pd 
from statsmodels.tsa.stattools import adfuller

# Loads the y variable (dependent variable) (effected variable)
yPath = "Data/TrendDailyFiltered.csv"
yDataset = pd.read_csv(yPath)

# Loads the x variable (independent variables) (caused variables)
XPath = "Data/MacroeconomicsDailyFiltered.csv"
XDataset = pd.read_csv(XPath)

# Defines a fuction to check rather the timeseries dataset is stationary or not stationary
def adfullerTest(series, signif=0.05, name='', verbose=False):
    r = adfuller(series, autolag='AIC')
    output = {'test_statistic':round(r[0], 4), 'pvalue':round(r[1], 4), 'n_lags':round(r[2], 4), 'n_obs':r[3]}
    p_value = output['pvalue'] 
    def adjust(val, length= 6): return str(val).ljust(length)

    print(f'    Augmented Dickey-Fuller Test on "{name}"', "\n   ", '-'*47)
    print(f' Null Hypothesis: Data has unit root. Non-Stationary.')
    print(f' Significance Level    = {signif}')
    print(f' Test Statistic        = {output["test_statistic"]}')
    print(f' No. Lags Chosen       = {output["n_lags"]}')

    for key,val in r[4].items():
        print(f' Critical value {adjust(key)} = {round(val, 3)}')

    if p_value <= signif:
        print(f" => P-Value = {p_value}. Rejecting Null Hypothesis.")
        print(f" => Series is Stationary.")
    else:
        print(f" => P-Value = {p_value}. Weak evidence to reject the Null Hypothesis.")
        print(f" => Series is Non-Stationary.")   
    
    return p_value

# Defines a fuction to check rather the timeseries dataset is stationary or not stationary (adding a differencing concept)
# The original source code program (weekly behavior)
def originalCheckStationary(dataframe):
    for diff_timess in range(5):
        non_Stationary_column=[]
        for name, column in dataframe.items():
            pvalue= adfullerTest(column, name=column.name)
            if pvalue >= 0.05:
                non_Stationary_column.append(column.name)
            print('\n')
            
        if len(non_Stationary_column) == 0:
            print(f"Completed {diff_timess} differencing step(s), the dataset is now stationary.")
            break
        else:
            print(f"Applying differencing (Step {diff_timess+1}) for: {non_Stationary_column}.")
            dataframe=dataframe.diff(periods=7).dropna()
    
    return dataframe

# The updated program sourced from the source code (weekly behavior)
def updatedCheckStationary(dataframe, maxDiff=1):
    dateColumnName = dataframe.columns[0]
    dateColumn = dataframe.iloc[:, 0]
    firstUpdatedDataframe = dataframe.iloc[:, 1:]
    constantColumns = [col for col in firstUpdatedDataframe.columns if firstUpdatedDataframe[col].nunique() == 1]
    
    if constantColumns:
        firstUpdatedDataframe = firstUpdatedDataframe.drop(columns=constantColumns)

    for diff_timess in range(maxDiff):
        non_Stationary_column = []
        
        # Checks stationarity of each column
        for name, column in firstUpdatedDataframe.items():
            pvalue = adfullerTest(column, name=column.name)
            if pvalue >= 0.05:
                non_Stationary_column.append(column.name)
            print('\n')
        
        if len(non_Stationary_column) == 0:
            print(f"Completed {diff_timess} differencing step(s), the dataset is now stationary.")
            break
        else:
            print(f"Applying differencing (Step {diff_timess+1}) for: {non_Stationary_column}.")
            firstUpdatedDataframe = firstUpdatedDataframe.diff(periods=7).dropna()
    
    # Final stationarity check after all differencing
    final_non_Stationary = []
    for name, column in firstUpdatedDataframe.items():
        pvalue = adfullerTest(column, name=column.name)
        if pvalue >= 0.05:
            final_non_Stationary.append(column.name)
        print('\n')

    firstUpdatedDataframe.insert(0, dateColumnName, dateColumn)

    return firstUpdatedDataframe, constantColumns, final_non_Stationary

yDatasetStationary, yDatasetStationaryConstantColumns, yDatasetStationaryNon = updatedCheckStationary(yDataset)
XDatasetStationary, XDatasetStationaryConstantColumns, XDatasetStationaryNon = updatedCheckStationary(XDataset)

print("The shape of stationary y dataset: ",yDatasetStationary.shape) # Shape [1813, 2] # Including 'Date' column
print("The number of constant column in y dataset: ",len(yDatasetStationaryConstantColumns)) # [0]
print("The number of non-stationay column after differencing process (y dataset): ",len(yDatasetStationaryNon)) # [0]

print("The shape of stationary X dataset: ",XDatasetStationary.shape) # Shape [1813, 9343] # Including 'Date' column
print("The number of constant column in X dataset: ",len(XDatasetStationaryConstantColumns)) # [48]
print("The number of non-stationay column after differencing process (X dataset): ",len(XDatasetStationaryNon)) # [476] 
# The number of features is 9342 because 1 column is a 'Date' column

# Saves the dataframe into a .csv file
yDatasetStationary .to_csv("Data/TrendFilteredStationary.csv", index=False)
XDatasetStationary .to_csv("Data/MacroeconomicsFilteredStationary.csv", index=False)

# Converts lists to DataFrame
XDatasetStationaryNondf = pd.DataFrame(XDatasetStationaryNon, columns=["Non-Stationary Columns"])
XDatasetStationaryConstantColumnsdf = pd.DataFrame(XDatasetStationaryConstantColumns, columns=["Constant Columns"])

# Saves the dataframe into a .csv file
XDatasetStationaryNondf.to_csv("Data/Macroeconomics(Non)Stationarylst.csv", index=False) # [476]
XDatasetStationaryConstantColumnsdf.to_csv("Data/MacroeconomicsConstantlst.csv", index=False) # [48]

# Still Non Stationary even though have already done first differencing

<h5> Deletes the columns that contain non-stationary time series sequences

In [ ]:
# Imports all necessary packages
import pandas as pd 

xDatasetStationaryRaw = pd.read_csv("Data/MacroeconomicsFilteredStationary.csv")
xDatasetNonStationaryColumnslst = pd.read_csv("Data/Macroeconomics(Non)Stationarylst.csv", header=None)

columnsRemovelst = xDatasetNonStationaryColumnslst[0].tolist()
# Removes the first row (the header of the column)
columnsRemovelst = columnsRemovelst[1:]

# Drop the non-stationary columns on the dataset
xDatasetStationaryRaw = xDatasetStationaryRaw.drop(columns=columnsRemovelst) # Shape [1813, 8867] # Including the 'Date' column

print(xDatasetStationaryRaw.shape)

# Saves a datafarme into a .csv file
xDatasetStationaryRaw.to_csv("Data/MacroeconomicsFilteredStationaryFinal.csv", index=False)

<h5> Implements the Granger Causality Test with a target value of trend (at timestep t)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Suppresses the warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Suppress warnings

# Loads datasets
yData = pd.read_csv("Data/TrendFilteredStationary.csv")
XData = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal.csv")

# Removes the first column (assumed to be 'Date')
yData = yData.iloc[:, 1:]
XData = XData.iloc[:, 1:]

# Merges DataFrames
data = pd.concat([yData, XData], axis=1)

# Extracts column names
yColumns = yData.columns.tolist()  # Effects (Y)
xColumns = XData.columns.tolist()  # Causes (X)

def grangersCausationMatrix(yColumns, xColumns, data, test='ssr_chi2test', maxlag=56, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=False): 
    resultsDictionary = {}
    count_ = 0

    for x in xColumns:
        for y in yColumns:
            test_result = grangercausalitytests(data[[y, x]], maxlag=maxlag, verbose=False)
            p_values = {f'Lag {lag}': round(test_result[lag][0][test][1], 4) for lag in lags if lag in test_result}
            resultsDictionary[(x, y)] = p_values
            count_ += 1

            if verbose:
                print(f'X = {x} → Y = {y}, P-Values = {p_values}')

    print(f'Tested {count_} variable combinations.')
    results_df = pd.DataFrame(resultsDictionary).T
    results_df.index.names = ['X (Cause)', 'Y (Effect)']

    return results_df

# Runs Granger causality test
grangerResultdf = grangersCausationMatrix(yColumns, xColumns, data, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=True)

# Displays the correct shape 
print(grangerResultdf.shape) # Shape [8866, 8]

# Saves results with X (rows) and Y (columns)
grangerResultdf.to_csv("Data/GrangerCausalityResult(Lag 01).csv", index=True)

<h5> Selects the columns where the number of lags with values below 0.05 exceeds the threshold (at timestep t)

In [ ]:
# Import necessary packages
import pandas as pd

def findGrangerCausalVariables(dataframe, significanceThreshold=0.05, minSignificantLags=5):
    causingVariables = []
    
    for _, row in dataframe.iterrows():
        significantLags = sum(row[2:] < significanceThreshold)
        if significantLags >= minSignificantLags:
            causingVariables.append(row["X (Cause)"])
    
    return list(dict.fromkeys(causingVariables))

causingVariables = findGrangerCausalVariables(pd.read_csv("Data/GrangerCausalityResult(Lag 01).csv"))

# Prints result
print("Variables that Granger-cause Y at a minimum of defined number of lags:")
for var in causingVariables:
    print(var)
print(len(causingVariables)) # Shape [1712, 1]

# Converts list to dataframe
causingVariablesdf = pd.DataFrame(causingVariables , columns=["Causing Variable (at timesteps 1)"])
causingVariablesdf.to_csv("Data/CausingVariableslst(Lag 01).csv", index=False)

<h5> Selects the column that Granger-causes the variable at timestep t for performing the Granger causality test (at timesteps t+7)

In [ ]:
# Imports all necessary packages
import pandas as pd 

xDatasetStationaryTimestep0 = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal.csv")
causingVariablesTimestep0 = pd.read_csv("Data/CausingVariableslst(Lag 01).csv", header=None)
# Shape [1813, 8867] [1713, 1]

causingVariablesTimestep0 = causingVariablesTimestep0[0].tolist()
# Removes the first row (the header of the column)
causingVariablesTimestep0 = causingVariablesTimestep0[1:]
# Inserts the 'Date' column to the list
causingVariablesTimestep0.insert(0, 'DATE')

# Takes the choosen columns on the list and adapts it on the dataset
xDatasetStationaryTimestep0 = xDatasetStationaryTimestep0[causingVariablesTimestep0] # Shape [1813, 1713]

# Saves a datafarme into a .csv file"Data/MacroeconomicsFilteredStationaryFinal(Lag 01).csv"
xDatasetStationaryTimestep0.to_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 01).csv", index=False)

<h5> Modifies the y variable instead the target variable at timesteps t, the target variable will be at timestep t+7

In [ ]:
# Imports all necessary packages
import pandas as pd

yDatasetTimestep0 = pd.read_csv("Data/TrendFilteredStationary.csv")

# Shift the 'Trend' column by 7 rows (upward)
yDatasetTimestep0['Trend'] = yDatasetTimestep0['Trend'].shift(-7)

# Drops the last 7 rows since they contain NaN
yDatasetTimestep0 = yDatasetTimestep0.iloc[:-7].reset_index(drop=True) # Shape [1806, 2]

# Saves a datafarme into a .csv file
yDatasetTimestep0.to_csv("Data/TrendFilteredStationary(Lag 01).csv", index=False)

<h5> Implements the Granger Causality Test with a target value of trend (at timestep t+7)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Suppresses the warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Suppress warnings

# Loads datasets
yData = pd.read_csv("Data/TrendFilteredStationary(Lag 01).csv")
XData = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 01).csv")

# Remove the last 7 rows from XData
XData = XData.iloc[:-7, :]

# Removes the first column (assumed to be 'Date')
yData = yData.iloc[:, 1:]
XData = XData.iloc[:, 1:]

# Merges DataFrames
data = pd.concat([yData, XData], axis=1)

# Extracts column names
yColumns = yData.columns.tolist()  # Effects (Y)
xColumns = XData.columns.tolist()  # Causes (X)

def grangersCausationMatrix(yColumns, xColumns, data, test='ssr_chi2test', maxlag=56, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=False): 
    resultsDictionary = {}
    count_ = 0

    for x in xColumns:
        for y in yColumns:
            test_result = grangercausalitytests(data[[y, x]], maxlag=maxlag, verbose=False)
            p_values = {f'Lag {lag}': round(test_result[lag][0][test][1], 4) for lag in lags if lag in test_result}
            resultsDictionary[(x, y)] = p_values
            count_ += 1

            if verbose:
                print(f'{count_}. X = {x} → Y = {y}, P-Values = {p_values}')

    print(f'Tested {count_} variable combinations.')
    results_df = pd.DataFrame(resultsDictionary).T
    results_df.index.names = ['X (Cause)', 'Y (Effect)']

    return results_df

# Runs Granger causality test
grangerResultdf = grangersCausationMatrix(yColumns, xColumns, data, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=True)

# Displays the correct shape 
print(grangerResultdf.shape) # Shape [1712, 8]

# Saves results with X (rows) and Y (columns)
grangerResultdf.to_csv("Data/GrangerCausalityResult(Lag 02).csv", index=True)

<h5> Selects the columns where the number of lags with values below 0.05 exceeds the threshold (at timestep t+7)

In [ ]:
# Import necessary packages
import pandas as pd

def findGrangerCausalVariables(dataframe, significanceThreshold=0.05, minSignificantLags=5):
    causingVariables = []
    
    for _, row in dataframe.iterrows():
        significantLags = sum(row[2:] < significanceThreshold)
        if significantLags >= minSignificantLags:
            causingVariables.append(row["X (Cause)"])
    
    return list(dict.fromkeys(causingVariables))

causingVariables = findGrangerCausalVariables(pd.read_csv("Data/GrangerCausalityResult(Lag 02).csv"))

# Prints result
print("Variables that Granger-cause Y at a minimum of defined number of lags:")
for var in causingVariables:
    print(var)
print(len(causingVariables)) # Shape [1090, 1]

# Converts list to dataframe
causingVariablesdf = pd.DataFrame(causingVariables , columns=["Causing Variable (at timesteps 2)"])
causingVariablesdf.to_csv("Data/CausingVariableslst(Lag 02).csv", index=False)

<h5> Selects the column that Granger-causes the variable at timestep t+7 for performing the Granger causality test (at timesteps t+14)

In [ ]:
# Imports all necessary packages
import pandas as pd 

xDatasetStationaryTimestep1 = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal.csv")
causingVariablesTimestep1 = pd.read_csv("Data/CausingVariableslst(Lag 02).csv", header=None)
# Shape [1813, 8867] [1091, 1]

causingVariablesTimestep1 = causingVariablesTimestep1[0].tolist()
# Removes the first row (the header of the column)
causingVariablesTimestep1 = causingVariablesTimestep1[1:]
# Inserts the 'Date' column to the list
causingVariablesTimestep1.insert(0, 'DATE')

# Takes the choosen columns on the list and adapts it on the dataset
xDatasetStationaryTimestep1 = xDatasetStationaryTimestep1[causingVariablesTimestep1] # Shape [1813, 1091]

# Saves a datafarme into a .csv file
xDatasetStationaryTimestep1.to_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 02).csv", index=False)

<h5> Modifies the y variable instead the target variable at timesteps t, the target variable will be at timestep t+14

In [ ]:
# Imports all necessary packages
import pandas as pd

yDatasetTimestep1 = pd.read_csv("Data/TrendFilteredStationary.csv")

# Shift the 'Trend' column by 14 rows (upward)
yDatasetTimestep1['Trend'] = yDatasetTimestep1['Trend'].shift(-14)

# Drops the last 14 rows since they contain NaN
yDatasetTimestep1 = yDatasetTimestep1.iloc[:-14].reset_index(drop=True) # Shape [1799, 2]

# Saves a datafarme into a .csv file
yDatasetTimestep1.to_csv("Data/TrendFilteredStationary(Lag 02).csv", index=False)

<h5> Implements the Granger Causality Test with a target value of trend (at timestep t+14)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Suppresses the warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Suppress warnings

# Loads datasets
yData = pd.read_csv("Data/TrendFilteredStationary(Lag 02).csv")
XData = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 02).csv")

# Remove the last 14 rows from XData
XData = XData.iloc[:-14, :]

# Removes the first column (assumed to be 'Date')
yData = yData.iloc[:, 1:]
XData = XData.iloc[:, 1:]

# Merges DataFrames
data = pd.concat([yData, XData], axis=1)

# Extracts column names
yColumns = yData.columns.tolist()  # Effects (Y)
xColumns = XData.columns.tolist()  # Causes (X)

def grangersCausationMatrix(yColumns, xColumns, data, test='ssr_chi2test', maxlag=56, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=False): 
    resultsDictionary = {}
    count_ = 0

    for x in xColumns:
        for y in yColumns:
            test_result = grangercausalitytests(data[[y, x]], maxlag=maxlag, verbose=False)
            p_values = {f'Lag {lag}': round(test_result[lag][0][test][1], 4) for lag in lags if lag in test_result}
            resultsDictionary[(x, y)] = p_values
            count_ += 1

            if verbose:
                print(f'{count_}. X = {x} → Y = {y}, P-Values = {p_values}')

    print(f'Tested {count_} variable combinations.')
    results_df = pd.DataFrame(resultsDictionary).T
    results_df.index.names = ['X (Cause)', 'Y (Effect)']

    return results_df

# Runs Granger causality test
grangerResultdf = grangersCausationMatrix(yColumns, xColumns, data, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=True)

# Displays the correct shape 
print(grangerResultdf.shape) # Shape [1090, 8]

# Saves results with X (rows) and Y (columns)
grangerResultdf.to_csv("Data/GrangerCausalityResult(Lag 03).csv", index=True)

<h5> Selects the columns where the number of lags with values below 0.05 exceeds the threshold (at timestep t+14)

In [ ]:
# Import necessary packages
import pandas as pd

def findGrangerCausalVariables(dataframe, significanceThreshold=0.05, minSignificantLags=5):
    causingVariables = []
    
    for _, row in dataframe.iterrows():
        significantLags = sum(row[2:] < significanceThreshold)
        if significantLags >= minSignificantLags:
            causingVariables.append(row["X (Cause)"])
    
    return list(dict.fromkeys(causingVariables))

causingVariables = findGrangerCausalVariables(pd.read_csv("Data/GrangerCausalityResult(Lag 03).csv"))

# Prints result
print("Variables that Granger-cause Y at a minimum of defined number of lags:")
for var in causingVariables:
    print(var)
print(len(causingVariables)) # Shape [655, 1]

# Converts list to dataframe
causingVariablesdf = pd.DataFrame(causingVariables , columns=["Causing Variable (at timesteps 3)"])
causingVariablesdf.to_csv("Data/CausingVariableslst(Lag 03).csv", index=False)

<h5> Selects the column that Granger-causes the variable at timestep t+14 for performing the Granger causality test (at timesteps t+21)

In [ ]:
# Imports all necessary packages
import pandas as pd 

xDatasetStationaryTimestep2 = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal.csv")
causingVariablesTimestep2 = pd.read_csv("Data/CausingVariableslst(Lag 03).csv", header=None)
# Shape [1813, 8867] [656, 1]

causingVariablesTimestep2 = causingVariablesTimestep2[0].tolist()
# Removes the first row (the header of the column)
causingVariablesTimestep2 = causingVariablesTimestep2[1:]
# Inserts the 'Date' column to the list
causingVariablesTimestep2.insert(0, 'DATE')

# Takes the choosen columns on the list and adapts it on the dataset
xDatasetStationaryTimestep2 = xDatasetStationaryTimestep2[causingVariablesTimestep2] # Shape [1813, 656]
print(xDatasetStationaryTimestep2.shape)

# Saves a datafarme into a .csv file
xDatasetStationaryTimestep2.to_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 03).csv", index=False)

<h5> Modifies the y variable instead the target variable at timesteps t, the target variable will be at timestep t+21

In [ ]:
# Imports all necessary packages
import pandas as pd

yDatasetTimestep2 = pd.read_csv("Data/TrendFilteredStationary.csv")

# Shift the 'Trend' column by 21 rows (upward)
yDatasetTimestep2['Trend'] = yDatasetTimestep2['Trend'].shift(-21)

# Drops the last 21 rows since they contain NaN
yDatasetTimestep2 = yDatasetTimestep2.iloc[:-21].reset_index(drop=True) # Shape [1792, 2]

# Saves a datafarme into a .csv file
yDatasetTimestep2.to_csv("Data/TrendFilteredStationary(Lag 03).csv", index=False)

<h5> Implements the Granger Causality Test with a target value of trend (at timestep t+21)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Suppresses the warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Suppress warnings

# Loads datasets
yData = pd.read_csv("Data/TrendFilteredStationary(Lag 03).csv")
XData = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 03).csv")

# Remove the last 21 rows from XData
XData = XData.iloc[:-21, :]

# Removes the first column (assumed to be 'Date')
yData = yData.iloc[:, 1:]
XData = XData.iloc[:, 1:]

# Merges DataFrames
data = pd.concat([yData, XData], axis=1)

# Extracts column names
yColumns = yData.columns.tolist()  # Effects (Y)
xColumns = XData.columns.tolist()  # Causes (X)

def grangersCausationMatrix(yColumns, xColumns, data, test='ssr_chi2test', maxlag=56, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=False): 
    resultsDictionary = {}
    count_ = 0

    for x in xColumns:
        for y in yColumns:
            test_result = grangercausalitytests(data[[y, x]], maxlag=maxlag, verbose=False)
            p_values = {f'Lag {lag}': round(test_result[lag][0][test][1], 4) for lag in lags if lag in test_result}
            resultsDictionary[(x, y)] = p_values
            count_ += 1

            if verbose:
                print(f'{count_}. X = {x} → Y = {y}, P-Values = {p_values}')

    print(f'Tested {count_} variable combinations.')
    results_df = pd.DataFrame(resultsDictionary).T
    results_df.index.names = ['X (Cause)', 'Y (Effect)']

    return results_df

# Runs Granger causality test
grangerResultdf = grangersCausationMatrix(yColumns, xColumns, data, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=True)

# Displays the correct shape 
print(grangerResultdf.shape)  # Shape [655, 8]

# Saves results with X (rows) and Y (columns)
grangerResultdf.to_csv("Data/GrangerCausalityResult(Lag 04).csv", index=True)

<h5> Selects the columns where the number of lags with values below 0.05 exceeds the threshold (at timestep t+21)

In [ ]:
# Import necessary packages
import pandas as pd

def findGrangerCausalVariables(dataframe, significanceThreshold=0.05, minSignificantLags=5):
    causingVariables = []
    
    for _, row in dataframe.iterrows():
        significantLags = sum(row[2:] < significanceThreshold)
        if significantLags >= minSignificantLags:
            causingVariables.append(row["X (Cause)"])
    
    return list(dict.fromkeys(causingVariables))

causingVariables = findGrangerCausalVariables(pd.read_csv("Data/GrangerCausalityResult(Lag 04).csv"))

# Prints result
print("Variables that Granger-cause Y at a minimum of defined number of lags:")
for var in causingVariables:
    print(var)
print(len(causingVariables)) # Shape [455, 1]

# Converts list to dataframe
causingVariablesdf = pd.DataFrame(causingVariables , columns=["Causing Variable (at timesteps 4)"])
causingVariablesdf.to_csv("Data/CausingVariableslst(Lag 04).csv", index=False)

<h5> Selects the column that Granger-causes the variable at timestep t+21 for performing the Granger causality test (at timesteps t+28)

In [ ]:
# Imports all necessary packages
import pandas as pd 

xDatasetStationaryTimestep3 = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal.csv")
causingVariablesTimestep3 = pd.read_csv("Data/CausingVariableslst(Lag 04).csv", header=None)
# Shape [1813, 8867] [456, 1]

causingVariablesTimestep3 = causingVariablesTimestep3[0].tolist()
# Removes the first row (the header of the column)
causingVariablesTimestep3 = causingVariablesTimestep3[1:]
# Inserts the 'Date' column to the list
causingVariablesTimestep3.insert(0, 'DATE')

# Takes the choosen columns on the list and adapts it on the dataset
xDatasetStationaryTimestep3 = xDatasetStationaryTimestep3[causingVariablesTimestep3] # Shape [1813, 456]
print(xDatasetStationaryTimestep3.shape)

# Saves a datafarme into a .csv file
xDatasetStationaryTimestep3.to_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 04).csv", index=False)

<h5> Modifies the y variable instead the target variable at timesteps t, the target variable will be at timestep t+28

In [ ]:
# Imports all necessary packages
import pandas as pd

yDatasetTimestep3 = pd.read_csv("Data/TrendFilteredStationary.csv")

# Shift the 'Trend' column by 28 rows (upward)
yDatasetTimestep3['Trend'] = yDatasetTimestep3['Trend'].shift(-28)

# Drops the last 21 rows since they contain NaN
yDatasetTimestep3 = yDatasetTimestep3.iloc[:-28].reset_index(drop=True) # Shape [1785, 2]

# Saves a datafarme into a .csv file
yDatasetTimestep3.to_csv("Data/TrendFilteredStationary(Lag 04).csv", index=False)

<h5> Implements the Granger Causality Test with a target value of trend (at timestep t+28)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Suppresses the warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Suppress warnings

# Loads datasets
yData = pd.read_csv("Data/TrendFilteredStationary(Lag 04).csv")
XData = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 04).csv")

# Remove the last 28 rows from XData
XData = XData.iloc[:-28, :]

# Removes the first column (assumed to be 'Date')
yData = yData.iloc[:, 1:]
XData = XData.iloc[:, 1:]

# Merges DataFrames
data = pd.concat([yData, XData], axis=1)

# Extracts column names
yColumns = yData.columns.tolist()  # Effects (Y)
xColumns = XData.columns.tolist()  # Causes (X)

def grangersCausationMatrix(yColumns, xColumns, data, test='ssr_chi2test', maxlag=56, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=False): 
    resultsDictionary = {}
    count_ = 0

    for x in xColumns:
        for y in yColumns:
            test_result = grangercausalitytests(data[[y, x]], maxlag=maxlag, verbose=False)
            p_values = {f'Lag {lag}': round(test_result[lag][0][test][1], 4) for lag in lags if lag in test_result}
            resultsDictionary[(x, y)] = p_values
            count_ += 1

            if verbose:
                print(f'{count_}. X = {x} → Y = {y}, P-Values = {p_values}')

    print(f'Tested {count_} variable combinations.')
    results_df = pd.DataFrame(resultsDictionary).T
    results_df.index.names = ['X (Cause)', 'Y (Effect)']

    return results_df

# Runs Granger causality test
grangerResultdf = grangersCausationMatrix(yColumns, xColumns, data, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=True)

# Displays the correct shape 
print(grangerResultdf.shape)  # Shape [455, 8]

# Saves results with X (rows) and Y (columns)
grangerResultdf.to_csv("Data/GrangerCausalityResult(Lag 05).csv", index=True)

<h5> Selects the columns where the number of lags with values below 0.05 exceeds the threshold (at timestep t+28)

In [ ]:
# Import necessary packages
import pandas as pd

def findGrangerCausalVariables(dataframe, significanceThreshold=0.05, minSignificantLags=5):
    causingVariables = []
    
    for _, row in dataframe.iterrows():
        significantLags = sum(row[2:] < significanceThreshold)
        if significantLags >= minSignificantLags:
            causingVariables.append(row["X (Cause)"])
    
    return list(dict.fromkeys(causingVariables))

causingVariables = findGrangerCausalVariables(pd.read_csv("Data/GrangerCausalityResult(Lag 05).csv"))

# Prints result
print("Variables that Granger-cause Y at a minimum of defined number of lags:")
for var in causingVariables:
    print(var)
print(len(causingVariables)) # Shape [316, 1]

# Converts list to dataframe
causingVariablesdf = pd.DataFrame(causingVariables , columns=["Causing Variable (at timesteps 5)"])
causingVariablesdf.to_csv("Data/CausingVariableslst(Lag 05).csv", index=False)

<h5> Selects the column that Granger-causes the variable at timestep t+28 for performing the Granger causality test (at timesteps t+35)

In [ ]:
# Imports all necessary packages
import pandas as pd 

xDatasetStationaryTimestep4 = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal.csv")
causingVariablesTimestep4 = pd.read_csv("Data/CausingVariableslst(Lag 05).csv", header=None)
# Shape [1813, 8867] [317, 1]

causingVariablesTimestep4 = causingVariablesTimestep4[0].tolist()
# Removes the first row (the header of the column)
causingVariablesTimestep4 = causingVariablesTimestep4[1:]
# Inserts the 'Date' column to the list
causingVariablesTimestep4.insert(0, 'DATE')

# Takes the choosen columns on the list and adapts it on the dataset
xDatasetStationaryTimestep4 = xDatasetStationaryTimestep4[causingVariablesTimestep4] # Shape [1813, 317]
print(xDatasetStationaryTimestep4.shape)

# Saves a datafarme into a .csv file
xDatasetStationaryTimestep4.to_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 05).csv", index=False)

<h5> Modifies the y variable instead the target variable at timesteps t, the target variable will be at timestep t+35

In [ ]:
# Imports all necessary packages
import pandas as pd

yDatasetTimestep4 = pd.read_csv("Data/TrendFilteredStationary.csv")

# Shift the 'Trend' column by 35 rows (upward)
yDatasetTimestep4['Trend'] = yDatasetTimestep4['Trend'].shift(-35)

# Drops the last 35 rows since they contain NaN
yDatasetTimestep4 = yDatasetTimestep4.iloc[:-35].reset_index(drop=True) # Shape [1778, 2]

# Saves a datafarme into a .csv file
yDatasetTimestep4.to_csv("Data/TrendFilteredStationary(Lag 05).csv", index=False)

<h5> Implements the Granger Causality Test with a target value of trend (at timestep t+35)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Suppresses the warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Suppress warnings

# Loads datasets
yData = pd.read_csv("Data/TrendFilteredStationary(Lag 05).csv")
XData = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 05).csv")

# Remove the last 35 rows from XData
XData = XData.iloc[:-35, :]

# Removes the first column (assumed to be 'Date')
yData = yData.iloc[:, 1:]
XData = XData.iloc[:, 1:]

# Merges DataFrames
data = pd.concat([yData, XData], axis=1)

# Extracts column names
yColumns = yData.columns.tolist()  # Effects (Y)
xColumns = XData.columns.tolist()  # Causes (X)

def grangersCausationMatrix(yColumns, xColumns, data, test='ssr_chi2test', maxlag=56, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=False): 
    resultsDictionary = {}
    count_ = 0

    for x in xColumns:
        for y in yColumns:
            test_result = grangercausalitytests(data[[y, x]], maxlag=maxlag, verbose=False)
            p_values = {f'Lag {lag}': round(test_result[lag][0][test][1], 4) for lag in lags if lag in test_result}
            resultsDictionary[(x, y)] = p_values
            count_ += 1

            if verbose:
                print(f'{count_}. X = {x} → Y = {y}, P-Values = {p_values}')

    print(f'Tested {count_} variable combinations.')
    results_df = pd.DataFrame(resultsDictionary).T
    results_df.index.names = ['X (Cause)', 'Y (Effect)']

    return results_df

# Runs Granger causality test
grangerResultdf = grangersCausationMatrix(yColumns, xColumns, data, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=True)

# Displays the correct shape 
print(grangerResultdf.shape)  # Shape [316, 8]

# Saves results with X (rows) and Y (columns)
grangerResultdf.to_csv("Data/GrangerCausalityResult(Lag 06).csv", index=True)

<h5> Selects the columns where the number of lags with values below 0.05 exceeds the threshold (at timestep t+35)

In [ ]:
# Import necessary packages
import pandas as pd

def findGrangerCausalVariables(dataframe, significanceThreshold=0.05, minSignificantLags=5):
    causingVariables = []
    
    for _, row in dataframe.iterrows():
        significantLags = sum(row[2:] < significanceThreshold)
        if significantLags >= minSignificantLags:
            causingVariables.append(row["X (Cause)"])
    
    return list(dict.fromkeys(causingVariables))

causingVariables = findGrangerCausalVariables(pd.read_csv("Data/GrangerCausalityResult(Lag 06).csv"))

# Prints result
print("Variables that Granger-cause Y at a minimum of defined number of lags:")
for var in causingVariables:
    print(var)
print(len(causingVariables)) # Shape [238, 1]

# Converts list to dataframe
causingVariablesdf = pd.DataFrame(causingVariables , columns=["Causing Variable (at timesteps 5)"])
causingVariablesdf.to_csv("Data/CausingVariableslst(Lag 06).csv", index=False)

<h5> Selects the column that Granger-causes the variable at timestep t+35 for performing the Granger causality test (at timesteps t+42)

In [ ]:
# Imports all necessary packages
import pandas as pd 

xDatasetStationaryTimestep5 = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal.csv")
causingVariablesTimestep5 = pd.read_csv("Data/CausingVariableslst(Lag 06).csv", header=None)
# Shape [1813, 8867] [239, 1]

causingVariablesTimestep5 = causingVariablesTimestep5[0].tolist()
# Removes the first row (the header of the column)
causingVariablesTimestep5 = causingVariablesTimestep5[1:]
# Inserts the 'Date' column to the list
causingVariablesTimestep5.insert(0, 'DATE')

# Takes the choosen columns on the list and adapts it on the dataset
xDatasetStationaryTimestep5 = xDatasetStationaryTimestep5[causingVariablesTimestep5] # Shape [1813, 239]
print(xDatasetStationaryTimestep5.shape)

# Saves a datafarme into a .csv file
xDatasetStationaryTimestep5.to_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 06).csv", index=False)

<h5> Modifies the y variable instead the target variable at timesteps t, the target variable will be at timestep t+42

In [ ]:
# Imports all necessary packages
import pandas as pd

yDatasetTimestep5 = pd.read_csv("Data/TrendFilteredStationary.csv")

# Shift the 'Trend' column by 42 rows (upward)
yDatasetTimestep5['Trend'] = yDatasetTimestep5['Trend'].shift(-42)

# Drops the last 42 rows since they contain NaN
yDatasetTimestep5 = yDatasetTimestep5.iloc[:-42].reset_index(drop=True) # Shape [1771, 2]

# Saves a datafarme into a .csv file
yDatasetTimestep5.to_csv("Data/TrendFilteredStationary(Lag 06).csv", index=False)

<h5> Implements the Granger Causality Test with a target value of trend (at timestep t+42)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Suppresses the warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Suppress warnings

# Loads datasets
yData = pd.read_csv("Data/TrendFilteredStationary(Lag 06).csv")
XData = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 06).csv")

# Remove the last 42 rows from XData
XData = XData.iloc[:-42, :]

# Removes the first column (assumed to be 'Date')
yData = yData.iloc[:, 1:]
XData = XData.iloc[:, 1:]

# Merges DataFrames
data = pd.concat([yData, XData], axis=1)

# Extracts column names
yColumns = yData.columns.tolist()  # Effects (Y)
xColumns = XData.columns.tolist()  # Causes (X)

def grangersCausationMatrix(yColumns, xColumns, data, test='ssr_chi2test', maxlag=56, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=False): 
    resultsDictionary = {}
    count_ = 0

    for x in xColumns:
        for y in yColumns:
            test_result = grangercausalitytests(data[[y, x]], maxlag=maxlag, verbose=False)
            p_values = {f'Lag {lag}': round(test_result[lag][0][test][1], 4) for lag in lags if lag in test_result}
            resultsDictionary[(x, y)] = p_values
            count_ += 1

            if verbose:
                print(f'{count_}. X = {x} → Y = {y}, P-Values = {p_values}')

    print(f'Tested {count_} variable combinations.')
    results_df = pd.DataFrame(resultsDictionary).T
    results_df.index.names = ['X (Cause)', 'Y (Effect)']

    return results_df

# Runs Granger causality test
grangerResultdf = grangersCausationMatrix(yColumns, xColumns, data, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=True)

# Displays the correct shape 
print(grangerResultdf.shape)  # Shape [238, 8]

# Saves results with X (rows) and Y (columns)
grangerResultdf.to_csv("Data/GrangerCausalityResult(Lag 07).csv", index=True)

<h5> Selects the columns where the number of lags with values below 0.05 exceeds the threshold (at timestep t+42)

In [ ]:
# Import necessary packages
import pandas as pd

def findGrangerCausalVariables(dataframe, significanceThreshold=0.05, minSignificantLags=5):
    causingVariables = []
    
    for _, row in dataframe.iterrows():
        significantLags = sum(row[2:] < significanceThreshold)
        if significantLags >= minSignificantLags:
            causingVariables.append(row["X (Cause)"])
    
    return list(dict.fromkeys(causingVariables))

causingVariables = findGrangerCausalVariables(pd.read_csv("Data/GrangerCausalityResult(Lag 07).csv"))

# Prints result
print("Variables that Granger-cause Y at a minimum of defined number of lags:")
for var in causingVariables:
    print(var)
print(len(causingVariables)) # Shape [166, 1]

# Converts list to dataframe
causingVariablesdf = pd.DataFrame(causingVariables , columns=["Causing Variable (at timesteps 7)"])
causingVariablesdf.to_csv("Data/CausingVariableslst(Lag 07).csv", index=False)

<h5> Selects the column that Granger-causes the variable at timestep t+42 for performing the Granger causality test (at timesteps t+49)

In [ ]:
# Imports all necessary packages
import pandas as pd 

xDatasetStationaryTimestep6 = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal.csv")
causingVariablesTimestep6 = pd.read_csv("Data/CausingVariableslst(Lag 07).csv", header=None)
# Shape [1813, 8867] [167, 1]

causingVariablesTimestep6 = causingVariablesTimestep6[0].tolist()
# Removes the first row (the header of the column)
causingVariablesTimestep6 = causingVariablesTimestep6[1:]
# Inserts the 'Date' column to the list
causingVariablesTimestep6.insert(0, 'DATE')

# Takes the choosen columns on the list and adapts it on the dataset
xDatasetStationaryTimestep6 = xDatasetStationaryTimestep6[causingVariablesTimestep6] # Shape [1813, 167]
print(xDatasetStationaryTimestep6.shape)

# Saves a datafarme into a .csv file
xDatasetStationaryTimestep6.to_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 07).csv", index=False)

<h5> Modifies the y variable instead the target variable at timesteps t, the target variable will be at timestep t+49

In [ ]:
# Imports all necessary packages
import pandas as pd

yDatasetTimestep6 = pd.read_csv("Data/TrendFilteredStationary.csv")

# Shift the 'Trend' column by 49 rows (upward)
yDatasetTimestep6['Trend'] = yDatasetTimestep6['Trend'].shift(-49)

# Drops the last 49 rows since they contain NaN
yDatasetTimestep6 = yDatasetTimestep6.iloc[:-49].reset_index(drop=True) # Shape [1764, 2]

# Saves a datafarme into a .csv file
yDatasetTimestep6.to_csv("Data/TrendFilteredStationary(Lag 07).csv", index=False)

<h5> Implements the Granger Causality Test with a target value of trend (at timestep t+49)

In [ ]:
# Imports necessary packages
import pandas as pd
from statsmodels.tsa.stattools import grangercausalitytests
import numpy as np

# Suppresses the warnings
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)  # Suppress warnings

# Loads datasets
yData = pd.read_csv("Data/TrendFilteredStationary(Lag 07).csv")
XData = pd.read_csv("Data/MacroeconomicsFilteredStationaryFinal(Lag 07).csv")

# Remove the last 49 rows from XData
XData = XData.iloc[:-49, :]

# Removes the first column (assumed to be 'Date')
yData = yData.iloc[:, 1:]
XData = XData.iloc[:, 1:]

# Merges DataFrames
data = pd.concat([yData, XData], axis=1)

# Extracts column names
yColumns = yData.columns.tolist()  # Effects (Y)
xColumns = XData.columns.tolist()  # Causes (X)

def grangersCausationMatrix(yColumns, xColumns, data, test='ssr_chi2test', maxlag=56, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=False): 
    resultsDictionary = {}
    count_ = 0

    for x in xColumns:
        for y in yColumns:
            test_result = grangercausalitytests(data[[y, x]], maxlag=maxlag, verbose=False)
            p_values = {f'Lag {lag}': round(test_result[lag][0][test][1], 4) for lag in lags if lag in test_result}
            resultsDictionary[(x, y)] = p_values
            count_ += 1

            if verbose:
                print(f'{count_}. X = {x} → Y = {y}, P-Values = {p_values}')

    print(f'Tested {count_} variable combinations.')
    results_df = pd.DataFrame(resultsDictionary).T
    results_df.index.names = ['X (Cause)', 'Y (Effect)']

    return results_df

# Runs Granger causality test
grangerResultdf = grangersCausationMatrix(yColumns, xColumns, data, lags=[7, 14, 21, 28, 35, 42, 49, 56], verbose=True)

# Displays the correct shape 
print(grangerResultdf.shape)  # Shape [166, 8]

# Saves results with X (rows) and Y (columns)
grangerResultdf.to_csv("Data/GrangerCausalityResult(Lag 08).csv", index=True)

<h5> Selects the columns where the number of lags with values below 0.05 exceeds the threshold (at timestep t+49)

In [ ]:
# Import necessary packages
import pandas as pd

def findGrangerCausalVariables(dataframe, significanceThreshold=0.05, minSignificantLags=5):
    causingVariables = []
    
    for _, row in dataframe.iterrows():
        significantLags = sum(row[2:] < significanceThreshold)
        if significantLags >= minSignificantLags:
            causingVariables.append(row["X (Cause)"])
    
    return list(dict.fromkeys(causingVariables))

causingVariables = findGrangerCausalVariables(pd.read_csv("Data/GrangerCausalityResult(Lag 08).csv"))

# Prints result
print("Variables that Granger-cause Y at a minimum of defined number of lags:")
for var in causingVariables:
    print(var)
print(len(causingVariables)) # Shape [123, 1]

# Converts list to dataframe
causingVariablesdf = pd.DataFrame(causingVariables , columns=["Causing Variable (at timesteps 8)"])
causingVariablesdf.to_csv("Data/CausingVariableslst(Lag 08).csv", index=False)